In [1]:
# IMPORT LIBRARIES

In [2]:
import pandas as pd
import sqlite3

In [3]:
# CONNECT DATABASE

In [4]:
conn = sqlite3.connect('ipl_data.db')
print("DataBase connected succesfully")

DataBase connected succesfully


In [5]:
# FIRST JOIN — MATCHES + DELIVERIES (RCB ONLY)

In [6]:
query = """
SELECT season, match_id, batting_team, bowling_team, over, ball, batter, batsman_runs
FROM matches
JOIN deliveries ON matches.id = deliveries.match_id 
WHERE team1 LIKE 'Royal Challengers%' or team2 LIKE 'Royal Challengers%'
LIMIT 15
"""
joined_data = pd.read_sql_query(query,conn)
print("Joined data (first 15 rows):")
display(joined_data)

Joined data (first 15 rows):


,season,match_id,batting_team,bowling_team,over,ball,batter,batsman_runs
0,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,0
1,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,0
2,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,0
3,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,0
4,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,0
5,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,6,BB McCullum,0
6,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,0,7,BB McCullum,0
7,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,1,BB McCullum,0
8,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,2,BB McCullum,4
9,2007/08,335982,Kolkata Knight Riders,Royal Challengers Bangalore,1,3,BB McCullum,4


In [7]:
# TOP 5 RCB BATSMEN BY TOTAL RUNS (JOIN)

In [8]:
query = """
SELECT batter,SUM(batsman_runs) as total_runs
FROM matches
JOIN deliveries ON matches.id = deliveries.match_id 
where batting_team LIKE 'Royal Challengers%'
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 5
"""
top_batsmen = pd.read_sql_query(query,conn)
print("Top 5 RCB batsmen by total runs:")
display(top_batsmen)

Top 5 RCB batsmen by total runs:


,batter,total_runs
0,V Kohli,8014
1,AB de Villiers,4510
2,CH Gayle,3175
3,F du Plessis,1636
4,GJ Maxwell,1266


In [9]:
# TOP 5 RCB BOWLERS BY TOTAL WICKETS (JOIN)

In [10]:
query = """
SELECT bowler,SUM(is_wicket) as total_wickets
FROM matches
JOIN deliveries ON matches.id = deliveries.match_id 
where bowling_team LIKE 'Royal Challengers%'
GROUP BY bowler
ORDER BY total_wickets DESC
LIMIT 5
"""
top_bowlers  = pd.read_sql_query(query,conn)
print("Top 5 RCB bowlers by total wickets:")
display(top_bowlers)

Top 5 RCB bowlers by total wickets:


,bowler,total_wickets
0,YS Chahal,143
1,HV Patel,108
2,Mohammed Siraj,86
3,R Vinay Kumar,83
4,Z Khan,56


In [11]:
# RUNS IN WINS vs LOSSES (JOIN + CASE)

In [12]:
query = """
SELECT 
    batter, 
    SUM(batsman_runs) as total_runs,
    SUM(CASE WHEN winner LIKE 'ROYAL CHALLENGERS%' THEN batsman_runs ELSE 0 END) as runs_in_wins,
    SUM(CASE WHEN winner NOT LIKE 'ROYAL CHALLENGERS%' THEN batsman_runs ELSE 0 END) as runs_in_loss
FROM matches
JOIN deliveries ON matches.id = deliveries.match_id
WHERE batting_team LIKE 'Royal Challengers%'
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 10
"""
runs_win_loss = pd.read_sql_query(query,conn)
print("Runs in wins vs losses (top 20):")
display(runs_win_loss)

Runs in wins vs losses (top 20):


,batter,total_runs,runs_in_wins,runs_in_loss
0,V Kohli,8014,4278,3709
1,AB de Villiers,4510,2616,1827
2,CH Gayle,3175,2303,861
3,F du Plessis,1636,905,731
4,GJ Maxwell,1266,785,481
5,JH Kallis,1132,544,588
6,KD Karthik,937,394,516
7,R Dravid,898,416,482
8,D Padikkal,884,514,370
9,RM Patidar,799,439,360


In [13]:
# LEFT JOIN — FIND MATCHES WITH NO DELIVERIES

In [14]:
query = """
SELECT *
FROM matches
LEFT JOIN deliveries ON matches.id = deliveries.match_id
where deliveries.match_id IS NULL
"""
missing = pd.read_sql_query(query,conn)
print(f"Matches with no deliveries data: {len(missing)}")
if len(missing) > 0:
    print(missing)
else:
    print("✅ All matches have deliveries data.")

Matches with no deliveries data: 0
✅ All matches have deliveries data.


In [15]:
# CLOSE CONNECTION

In [16]:
conn.close()
print("Connection closed.")

Connection closed.


## Day 6 Summary: JOINs

 Completed
- Joined matches and deliveries tables on match_id
- Calculated top 5 RCB batsmen by total runs using JOIN + GROUP BY
- Calculated top 5 RCB bowlers by total wickets using JOIN + WHERE
- Split batting performance by match outcome (wins vs losses) using JOIN + CASE
- Tested LEFT JOIN to check for missing data

 Key Queries Learned
- INNER JOIN / JOIN for combining tables
- LEFT JOIN for finding missing records
- JOIN + GROUP BY for cross-table aggregation
- JOIN + CASE WHEN for conditional grouping across tables

 Verification
✅ Top batsman: Virat Kohli — 8,014 runs (matches Pandas result)
✅ Runs in wins vs losses pattern matches EDA project

 Next
Day 7 — Review & Polish: Organize best queries into a clean .sql file